# 01 - EDA Sparkov (`fraudTrain.csv` uniquement)

Ce notebook sert à explorer le fichier **`fraudTrain.csv`** seulement.

Règles importantes :
- ne jamais utiliser `fraudTest.csv` ici ;
- ne faire aucun entraînement de modèle ;
- ne supprimer aucune donnée ;
- ne pas modifier les choix définis dans `P2_Data_Scientist.md`.

L'objectif est de comprendre les données avant le preprocessing et le benchmark.

## 1. Imports

On importe uniquement les bibliothèques utiles pour lire les données, calculer des statistiques simples et produire quelques graphiques.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "ml":
    PROJECT_ROOT = PROJECT_ROOT.parent

ML_SRC = PROJECT_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.append(str(ML_SRC))

from data.load_data import load_development_dataset, print_dataset_summary

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8")

## 2. Chargement de `fraudTrain.csv`

`fraudTrain.csv` est le fichier réservé au développement, au benchmark et à la calibration. Le fichier `fraudTest.csv` est réservé au test final et ne doit pas être utilisé dans cette analyse.

Place le fichier brut dans `data/raw/fraudTrain.csv` avant d'exécuter la cellule.

In [ ]:
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "fraudTrain.csv"

df, train_summary = load_development_dataset(TRAIN_PATH)
print_dataset_summary(train_summary)

## 3. Aperçu avec `df.head()`

Les premières lignes permettent de voir rapidement les colonnes disponibles et le format général des valeurs.

In [ ]:
df.head()

## 4. Dimensions avec `shape`

`shape` donne le nombre de lignes et le nombre de colonnes du dataset.

In [ ]:
df.shape

## 5. Structure avec `info()`

`info()` indique le type de chaque colonne et le nombre de valeurs non manquantes.

In [ ]:
df.info()

## 6. Statistiques descriptives avec `describe()`

`describe()` résume les colonnes numériques : moyenne, écart-type, minimum, quartiles et maximum.

In [ ]:
df.describe()

## 7. Types des colonnes

Cette étape aide à distinguer les colonnes numériques, textuelles et temporelles avant tout preprocessing.

In [ ]:
df.dtypes.sort_values()

## 8. Valeurs manquantes

On compte les valeurs absentes colonne par colonne. À ce stade, on ne les remplace pas et on ne supprime aucune ligne.

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

pd.DataFrame({
    "valeurs_manquantes": missing_values,
    "pourcentage": missing_percent,
}).query("valeurs_manquantes > 0")

## 9. Doublons

On vérifie s'il existe des lignes totalement identiques. On ne supprime rien dans ce notebook.

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_percent = duplicate_count / len(df) * 100

duplicate_count, duplicate_percent

## 10. Distribution de `is_fraud`

`is_fraud` est la cible : `0` signifie transaction légitime, `1` signifie fraude.

In [ ]:
fraud_distribution = df["is_fraud"].value_counts().sort_index()
fraud_distribution

## 11. Nombre et pourcentage exact de fraudes

Cette cellule calcule explicitement le nombre de fraudes et leur pourcentage dans `fraudTrain.csv`.

In [ ]:
fraud_count = int(df["is_fraud"].sum())
total_count = len(df)
fraud_percent = fraud_count / total_count * 100

print(f"Nombre total de transactions : {total_count}")
print(f"Nombre exact de fraudes : {fraud_count}")
print(f"Pourcentage exact de fraudes : {fraud_percent:.6f} %")

## 12. Statistiques sur `amt`

`amt` correspond au montant de la transaction. On observe sa dispersion, ses quartiles et ses valeurs extrêmes.

In [ ]:
df["amt"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

## 13. Distribution de `category`

La catégorie décrit le type de transaction. On regarde les catégories les plus fréquentes et leur taux de fraude.

In [ ]:
category_analysis = df.groupby("category").agg(
    transactions=("is_fraud", "size"),
    fraudes=("is_fraud", "sum"),
    taux_fraude=("is_fraud", "mean"),
).sort_values("transactions", ascending=False)

category_analysis

## 14. Analyse de `gender`

Le genre est conservé comme feature selon le document P2. Ici, on observe simplement la répartition et le taux de fraude par groupe.

In [ ]:
gender_analysis = df.groupby("gender").agg(
    transactions=("is_fraud", "size"),
    fraudes=("is_fraud", "sum"),
    taux_fraude=("is_fraud", "mean"),
).sort_values("transactions", ascending=False)

gender_analysis

## 15. Analyse de `state`

On observe les États avec le plus de transactions et les taux de fraude associés. Attention : un taux élevé sur peu de transactions peut être instable.

In [ ]:
state_analysis = df.groupby("state").agg(
    transactions=("is_fraud", "size"),
    fraudes=("is_fraud", "sum"),
    taux_fraude=("is_fraud", "mean"),
).sort_values("transactions", ascending=False)

state_analysis.head(15)

## 16. Analyse temporelle avec `trans_date_trans_time`

On convertit la colonne de date en datetime dans une copie de travail pour analyser les heures, jours et mois. Le DataFrame original `df` reste inchangé.

In [ ]:
df_time = df.copy()
df_time["trans_date_trans_time"] = pd.to_datetime(df_time["trans_date_trans_time"])
df_time["hour"] = df_time["trans_date_trans_time"].dt.hour
df_time["day_of_week"] = df_time["trans_date_trans_time"].dt.day_name()
df_time["month"] = df_time["trans_date_trans_time"].dt.to_period("M")

print(f"Date minimale : {df_time['trans_date_trans_time'].min()}")
print(f"Date maximale : {df_time['trans_date_trans_time'].max()}")

df_time.groupby("hour").agg(
    transactions=("is_fraud", "size"),
    taux_fraude=("is_fraud", "mean"),
)

## 17. Comparaison fraude / non-fraude pour le montant

On compare les montants selon la classe. Cela aide à voir si les fraudes ont des montants typiquement différents.

In [ ]:
df.groupby("is_fraud")["amt"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])

## 18. Comparaison fraude / non-fraude selon l'heure

On compare la répartition horaire des transactions légitimes et frauduleuses.

In [ ]:
hour_class_analysis = df_time.groupby(["hour", "is_fraud"]).size().unstack(fill_value=0)
hour_class_analysis["taux_fraude"] = hour_class_analysis.get(1, 0) / hour_class_analysis.sum(axis=1)
hour_class_analysis

## 19. Quelques visualisations simples

Ces graphiques donnent une lecture visuelle rapide : déséquilibre de la cible, montants, catégories, genre, États et heure de transaction.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

df["is_fraud"].value_counts().sort_index().plot(kind="bar", ax=axes[0, 0], title="Distribution de is_fraud")
axes[0, 0].set_xlabel("is_fraud")
axes[0, 0].set_ylabel("Nombre de transactions")

df["amt"].plot(kind="hist", bins=80, ax=axes[0, 1], title="Distribution de amt")
axes[0, 1].set_xlabel("Montant")

df.groupby("is_fraud")["amt"].median().plot(kind="bar", ax=axes[0, 2], title="Montant médian par classe")
axes[0, 2].set_xlabel("is_fraud")
axes[0, 2].set_ylabel("Montant médian")

df["category"].value_counts().head(10).plot(kind="bar", ax=axes[1, 0], title="Top 10 catégories")
axes[1, 0].set_xlabel("Catégorie")
axes[1, 0].set_ylabel("Nombre de transactions")

df.groupby("gender")["is_fraud"].mean().plot(kind="bar", ax=axes[1, 1], title="Taux de fraude par genre")
axes[1, 1].set_xlabel("Genre")
axes[1, 1].set_ylabel("Taux de fraude")

df_time.groupby("hour")["is_fraud"].mean().plot(kind="line", marker="o", ax=axes[1, 2], title="Taux de fraude par heure")
axes[1, 2].set_xlabel("Heure")
axes[1, 2].set_ylabel("Taux de fraude")

plt.tight_layout()
plt.show()

## Conclusions de l'EDA

La cellule suivante calcule les observations principales à partir de `fraudTrain.csv`. Elle ne contient pas de valeur inventée : les phrases sont générées à partir des données réellement chargées.

In [ ]:
from IPython.display import Markdown, display

top_category = category_analysis["transactions"].idxmax()
top_category_count = int(category_analysis.loc[top_category, "transactions"])
highest_fraud_category = category_analysis["taux_fraude"].idxmax()
highest_fraud_category_rate = category_analysis.loc[highest_fraud_category, "taux_fraude"] * 100

median_amt_legit = df.loc[df["is_fraud"] == 0, "amt"].median()
median_amt_fraud = df.loc[df["is_fraud"] == 1, "amt"].median()

peak_fraud_hour = df_time.groupby("hour")["is_fraud"].mean().idxmax()
peak_fraud_hour_rate = df_time.groupby("hour")["is_fraud"].mean().max() * 100

missing_total = int(df.isna().sum().sum())

conclusions = f"""
- Le fichier `fraudTrain.csv` contient **{total_count:,} transactions** et **{df.shape[1]} colonnes**.
- La cible est très déséquilibrée : **{fraud_count:,} fraudes**, soit **{fraud_percent:.6f} %** des transactions.
- Le dataset contient **{missing_total:,} valeurs manquantes** au total.
- Le nombre de lignes totalement dupliquées est **{duplicate_count:,}**, soit **{duplicate_percent:.6f} %** du dataset.
- La catégorie la plus fréquente est **`{top_category}`** avec **{top_category_count:,} transactions**.
- La catégorie avec le taux de fraude le plus élevé est **`{highest_fraud_category}`** avec **{highest_fraud_category_rate:.6f} %**.
- Le montant médian des transactions légitimes est **{median_amt_legit:.2f}**, contre **{median_amt_fraud:.2f}** pour les transactions frauduleuses.
- L'heure avec le taux de fraude le plus élevé est **{peak_fraud_hour}h**, avec **{peak_fraud_hour_rate:.6f} %**.
- Ces observations servent uniquement à comprendre les données ; elles ne constituent pas encore du preprocessing ni du Machine Learning.
"""

display(Markdown(conclusions))